# T31 / E14 — Bậc thang kích thước mô hình đọc

**Notebook này chạy BA LẦN, mỗi lần một phiên Kaggle riêng.** Không phải ba lần cho vui: mục 5
của `CLAUDE.md` đo được card T4 bị **hạ xung 10–15 %** sau vài phút chạy liên tục. Ba mô hình
chạy nối nhau trong một phiên thì phần hạ xung sẽ bị tính nhầm thành khác biệt giữa các mô hình
— mà chi phí mỗi mẫu chính là một nửa câu hỏi của E14.

## Chỉ phải chạy HAI lần, không phải ba

Nấc 7B đã có sẵn từ E02 và E03 — cùng bộ dữ liệu, cùng cách chia đoạn, cùng nhóm đặc trưng. Nên
notebook này chỉ chạy hai nấc còn lại:

| Nấc | Cần chạy? | Lấy số từ đâu |
|---|---|---|
| Qwen2.5-7B | **không** | E02 (0,7451) và E03 (0,7567), đã có |
| Qwen2.5-3B | **có** | phiên 1 của notebook này |
| Qwen2.5-1.5B | **có** | phiên 2 |

## Hai câu hỏi E14 trả lời

1. **Câu cũ:** phương pháp có chạy được trên phần cứng nhỏ hơn không, và mất bao nhiêu điểm khi
   thu mô hình đọc lại. Đây là trục chi phí của CH2.
2. **Câu mới, đáng giá hơn, do T30 sinh ra:** E13 đo được nhóm chunk-aware **không chuyển** giữa
   Qwen2.5-7B và Sailor2-8B — hai mô hình khác họ huấn luyện. Bậc thang này hỏi nó có chuyển
   giữa các **cỡ của cùng một họ** không. Đây là phép thử nhẹ hơn hẳn: nếu nó cũng không chuyển
   ở đây thì kết luận "hình dạng phân bố gắn với từng mô hình cụ thể" mạnh lên rất nhiều.

Vì câu thứ hai, mỗi cỡ có **hai** cấu hình: một chunk-aware và một mốc lookback gộp của chính
nó. T30 dạy bài này bằng một kết luận sai — so chunk-aware của Sailor2 với mốc của Qwen thì
chênh lệch trộn hai biến, không tách được.

## Lớp tràn số vẫn phải đo, không suy ra được

Qwen2.5-7B hỏng đúng lớp 27. Nhưng 3B và 1.5B là mô hình khác, số lớp khác. Và T30 cho thấy
chuyện này **không** suy ra được: Sailor2 hỏng theo kiểu hoàn toàn khác — cả mạng hỏng trên
0,7 % mẫu thay vì một lớp hỏng trên mọi mẫu.

Ô 5 đo trước rồi tự ghi vào **cả hai** cấu hình của cỡ đang chạy. `exclude_layers` để trống trong
repo là cố ý.

Tin tốt: hai mô hình này nhỏ hơn Sailor2 nhiều nên lượt mốc `bfloat16` sẽ vừa bộ nhớ, tức lần
này có cả phần kiểm "các lớp còn sống có bị bóp méo không" mà Sailor2 không cho được.

## Chi phí ước tính

7B đo được 528 ms/mẫu. Suy ra theo số tham số thì 3B khoảng 250 ms (~29 phút cho 7.000 mẫu) và
1.5B khoảng 150 ms (~18 phút). Cộng tải mô hình và ô dò kiểu số, mỗi phiên khoảng **45 phút**.

**Chấm điểm chạy ở máy cá nhân**, theo quy tắc chốt ở T23.


## Chuẩn bị

Ô 2 là **dòng duy nhất phải sửa** giữa hai phiên. Ô 4 sẽ bắt nếu quên đổi.

In [ ]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    print("$", " ".join(str(a) for a in args))
    result = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    print(result.stdout.strip())
    if result.returncode:
        print(result.stderr.strip())
        raise SystemExit(f"lệnh hỏng: {' '.join(str(a) for a in args)}")
    return result.stdout


if REPO_DIR.exists():
    run("git", "fetch", "--all", cwd=REPO_DIR)
    run("git", "reset", "--hard", "origin/main", cwd=REPO_DIR)
else:
    run("git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR))

os.chdir(REPO_DIR)
run("git", "log", "-1", "--format=%h %s")

In [ ]:
# Ô 2 — CHỌN CỠ MÔ HÌNH. ĐÂY LÀ DÒNG DUY NHẤT PHẢI SỬA GIỮA HAI PHIÊN.
#
#   Phiên 1:  CO = "3B"
#   Phiên 2:  CO = "1.5B"
#
# Quên đổi thì ô 3 sẽ bắt được và dừng lại, chứ không âm thầm chạy lại cùng một mô hình.
CO = "3B"

BAC_THANG = {
    "3B": {
        "mo_hinh": "Qwen/Qwen2.5-3B-Instruct",
        "chunk": "configs/e14_qwen3b_vihallu.yaml",
        "moc": "configs/e14_baseline_lookback_qwen3b.yaml",
        "uoc_tinh": "khoảng 29 phút",
    },
    "1.5B": {
        "mo_hinh": "Qwen/Qwen2.5-1.5B-Instruct",
        "chunk": "configs/e14_qwen15b_vihallu.yaml",
        "moc": "configs/e14_baseline_lookback_qwen15b.yaml",
        "uoc_tinh": "khoảng 18 phút",
    },
}
if CO not in BAC_THANG:
    raise SystemExit(f"CO phai la mot trong {list(BAC_THANG)}, dang la {CO!r}")
NAC = BAC_THANG[CO]

print("=" * 78)
print(f"  PHIEN NAY CHAY NAC: {CO}")
print(f"  mo hinh doc       : {NAC['mo_hinh']}")
print(f"  cau hinh chunk    : {NAC['chunk']}")
print(f"  cau hinh moc      : {NAC['moc']}")
print(f"  uoc tinh trich    : {NAC['uoc_tinh']}")
print("=" * 78)

In [ ]:
# Ô 3 — cài đặt. bitsandbytes cần cho lượng tử hóa 4 bit.
!pip install -q --no-deps -e .
!pip install -q bitsandbytes

In [ ]:
# Ô 4 — TIỀN KIỂM. Vài giây, chạy trước mọi thứ.
#
# Ngoài chuỗi phụ thuộc thường lệ, ô này còn bắt một lỗi riêng của T31: chạy lại đúng cỡ mô hình
# đã chạy phiên trước. Notebook này chạy hai lần với hai giá trị CO khác nhau, và quên đổi CO là
# cách dễ nhất để đốt 30 phút GPU vào thứ đã có.
import importlib.util
import sys
from pathlib import Path

sys.path.insert(0, "src")
from vihallulens.config import extraction_hash, load_config
from vihallulens.data.paths import find_raw_dir

cfg = load_config(NAC["chunk"])
problems = []

# --- PHẢI CÓ SẴN -------------------------------------------------------------------------------
packages = ("torch", "transformers", "bitsandbytes", "pandas", "accelerate")
absent = [name for name in packages if importlib.util.find_spec(name) is None]
trang_thai = f"THIEU {absent}" if absent else f"du ca {list(packages)}"
print(f"  goi phai co san   : {trang_thai}")
if absent:
    problems.append(f"thieu goi {absent}")

try:
    raw = find_raw_dir()
    files = sorted(p.name for p in Path(raw).glob("vihallu*"))
    print(f"  du lieu tho       : {raw}")
    print(f"  file vihallu tho  : {files or 'KHONG CO'}")
    if not files:
        problems.append("khong thay file vihallu nao trong du lieu tho")
except Exception as error:
    print(f"  du lieu tho       : KHONG TIM THAY ({error})")
    problems.append("chua mount dataset du lieu tho")

# --- ĐÃ CHẠY CỠ NÀY CHƯA ------------------------------------------------------------------------
xong = sorted(Path("/kaggle/working").glob("ket_qua_t31_*"))
if xong:
    print(f"  thu muc ket qua da co: {[p.name for p in xong]}")
    if any(p.name == f"ket_qua_t31_{CO.replace('.', '_')}" for p in xong):
        problems.append(f"co ve da chay nac {CO} roi trong phien nay — doi CO truoc khi chay")

print(f"  nac dang chay     : {CO}  ({cfg.extractor.model_name})")
print(f"  exclude_layers    : {cfg.extractor.exclude_layers}  <- o 6 se ghi de")

chain = [
    ("o 5", "data/interim/vihallu_{train,dev,test}.parquet", "normalize_data + split_data"),
    ("o 6", "exclude_layers trong CA HAI cau hinh cua nac nay", "compare_dtypes"),
    ("o 8", "data/processed/vihallu_{split}_<hash>.jsonl", "extract_features"),
]
print("  chuoi tu tao:")
for cell, target, maker in chain:
    print(f"    {cell:<5} {target:<52} <- {maker}")

if problems:
    raise SystemExit("TIEN KIEM HONG: " + "; ".join(problems))
print("\nTien kiem dat.")

In [ ]:
# Ô 5 — chuẩn bị dữ liệu và kiểm môi trường. Khoảng 2 phút, CPU.
#
# tests/test_attention_hook.py chay o day chu khong chi o may ca nhan: notebook cai bang
# --no-deps nen dung ban transformers cua Kaggle. Muc 5 CLAUDE.md canh bao viec hook co nhan
# duoc attn_weights hay khong PHU THUOC PHIEN BAN.
!python scripts/probe_env.py
!python scripts/normalize_data.py --dataset vihallu
!python scripts/split_data.py --only vihallu
!python -m pytest tests/test_attention_hook.py tests/test_drop_nonfinite.py -q

## Dò lớp tràn số — vẫn phải đo

Qwen2.5-7B hỏng đúng lớp 27, nhưng cỡ này là mô hình khác và T30 cho thấy chuyện này **không suy
ra được**: Sailor2 hỏng theo kiểu hoàn toàn khác Qwen — cả mạng hỏng trên 0,7 % mẫu thay vì một
lớp hỏng trên mọi mẫu.

Ô 6 ghi kết quả vào **cả hai** cấu hình của cỡ đang chạy. Ô 7 kiểm hai hash có trùng nhau không —
lệch là mốc lookback sẽ đòi trích lại từ đầu thay vì dùng lại shard.

In [ ]:
# Ô 6 — DÒ LỚP TRÀN SỐ. Khoảng 5 phút GPU. BẮT BUỘC chạy trước ô 8.
#
# Qwen2.5-7B hong dung lop 27. Nhung co nay la mo hinh khac, so lop khac. Va T30 cho thay chuyen
# nay KHONG suy ra duoc: Sailor2 hong theo kieu hoan toan khac Qwen.
#
# Hai mo hinh nay nho hon Sailor2 nhieu nen luot moc bfloat16 se vua bo nho, tuc lan nay co ca
# phan kiem "cac lop con song co bi bop meo khong" ma Sailor2 khong cho duoc.
import ast
import os
import re
from pathlib import Path

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

proc = subprocess.run(
    ["python", "scripts/compare_dtypes.py", "--model", NAC["mo_hinh"],
     "--per-dataset", "10", "--reference", "bfloat16"],
    capture_output=True, text=True, env={**os.environ},
)
print(proc.stdout[-8000:])
if proc.returncode:
    print(proc.stderr[-4000:])
    raise SystemExit("do kieu so hong")

found = re.search(r"^EXCLUDE_LAYERS=(\[.*\])$", proc.stdout, flags=re.MULTILINE)
if not found:
    raise SystemExit("khong thay dong EXCLUDE_LAYERS= trong output")
bad = sorted(set(ast.literal_eval(found.group(1))))
print()
print(f"  Lop tran so do duoc: {bad if bad else 'KHONG CO'}")

# Ghi vao CA HAI cau hinh cua nac nay. Thieu mot cai thi hash trich cua chung lech nhau va moc
# lookback se doi trich lai tu dau thay vi dung lai shard.
for duong_dan in (NAC["chunk"], NAC["moc"]):
    path = Path(duong_dan)
    text = path.read_text(encoding="utf-8")
    patched = re.sub(r"^  exclude_layers: \[\]$", f"  exclude_layers: {bad}", text, count=1,
                     flags=re.MULTILINE)
    if patched == text:
        raise SystemExit(f"khong tim thay dong 'exclude_layers: []' trong {duong_dan}")
    path.write_text(patched, encoding="utf-8")
    print(f"  Da ghi: {duong_dan}")

if not bad:
    print()
    print("  KHONG lop nao tran so. Voi Qwen2.5-7B thi lop cuoi luon tran, nen ket qua nay dang")
    print("  ngo — nhung cung co the that voi mo hinh nho hon. Doc bang per-layer o tren truoc")
    print("  khi chay tiep. Neu bang do sach that thi day la mot phat hien dang ghi.")
print()
print("  NHO commit lai HAI file config nay sau khi chay xong.")

In [ ]:
# Ô 7 — cổng kiểm trước khi tiêu GPU. Vài giây, CPU.
import sys

sys.path.insert(0, "src")
from importlib import reload

import vihallulens.config as config_module

reload(config_module)
chunk = config_module.load_config(NAC["chunk"])
moc = config_module.load_config(NAC["moc"])
h_chunk = config_module.extraction_hash(chunk)
h_moc = config_module.extraction_hash(moc)

print(f"  nac               : {CO}  ({chunk.extractor.model_name})")
print(f"  exclude_layers    : {chunk.extractor.exclude_layers}")
print(f"  hash chunk-aware  : {h_chunk}")
print(f"  hash moc lookback : {h_moc}")

if h_chunk != h_moc:
    raise SystemExit(
        "HAI HASH KHAC NHAU. Moc lookback se doi trich lai tu dau thay vi dung lai shard.\n"
        "Nguyen nhan gan nhu chac chan la o 6 chi ghi duoc vao mot trong hai file config."
    )
print("  -> Hai hash trung nhau, moc lookback se dung lai dung shard nay.")

## Trích đặc trưng

**Đọc gì trong lúc chạy:** dòng `lỗi` phải là 0, và dòng `LỚP TRÀN SỐ` nếu xuất hiện thì phải
đọc kỹ — nó liệt kê lớp nào tràn và bao nhiêu mẫu. Nếu nó liệt kê *mọi* lớp thì đó là kiểu hỏng
của Sailor2, xử lý bằng cách bỏ mẫu chứ không bỏ lớp.

In [ ]:
# Ô 8 — trích đặc trưng. Xem ước tính ở ô 2. Chạy lại được, có lưu tiến độ.
import os

# Tat thanh tien trinh nap trong so: no in ra hang nghin dong va day log vuot gioi han cua
# trinh xem, cat mat dung phan can doc.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

for split in ("train", "dev", "test"):
    code = os.system(f"python scripts/extract_features.py --config {NAC['chunk']} --split {split}")
    if code:
        raise SystemExit(f"trich tap {split} hong, ma loi {code}")

In [ ]:
# Ô 9 — soi shard. Vài giây, CPU. KHÔNG dừng notebook.
#
# Khong raise du shard co nan: mot shard hong la thu CAN dem ve nhat de chan doan. Bai hoc T30.
import subprocess

proc = subprocess.run(["python", "scripts/inspect_shard.py", "--config", NAC["chunk"]],
                      capture_output=True, text=True)
print(proc.stdout[-8000:])
if proc.returncode:
    print(proc.stderr[-2000:])
print()
print("  DU SHARD CO NAN HAY KHONG, VAN CHAY O 10 DE MANG VE.")

## Chấm điểm — KHÔNG chạy ở đây

Quy tắc chốt ở T23: mọi phép so sánh phải chấm trên **cùng một máy**, vì điểm dev lệch tới 0,0075
giữa Kaggle và máy cá nhân do bộ giải tối ưu hội tụ khác nhau.

In [ ]:
# Ô 10 — lấy kết quả về. Vài giây.
import shutil
import sys
from pathlib import Path

sys.path.insert(0, "src")
from vihallulens.config import load_config

cfg = load_config(NAC["chunk"])
run = extraction_hash(cfg)

out = Path(f"/kaggle/working/ket_qua_t31_{CO.replace('.', '_')}")
out.mkdir(exist_ok=True)
for split in ("train", "dev", "test"):
    src = Path(f"data/processed/vihallu_{split}_{run}.jsonl")
    if src.exists():
        shutil.copy(src, out / src.name)
for duong_dan in (NAC["chunk"], NAC["moc"]):
    shutil.copy(duong_dan, out / Path(duong_dan).name)

for f in sorted(out.iterdir()):
    print(f"  {f.name:<44} {f.stat().st_size / 1e6:>8.1f} MB")

print(f"""
Tai het thu muc {out.name} ve may, dat vao:
  *.jsonl  ->  data/processed/
  *.yaml   ->  configs/   (GHI DE — chung mang exclude_layers da do)

XONG NAC NAY ROI THI:
  - Neu dang o nac 3B  : mo phien Kaggle MOI, doi CO = "1.5B" o o 2, chay lai tu dau.
  - Neu da xong ca hai : bao lai de cham diem o may ca nhan.

DUNG chay nac thu hai trong cung phien nay. T4 ha xung 10-15 % sau vai phut chay lien tuc,
va chi phi moi mau chinh la mot nua cau hoi cua E14.
""")